In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import shutil
import glob

# 1. Mount Drive if not already mounted
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 2. Automatically search for any zip file containing 'archive' in Google Drive
found_zip = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'archive' in file and file.endswith('.zip'):
            found_zip = os.path.join(root, file)
            break
    if found_zip:
        break

if found_zip:
    print(f"✅ Found your dataset at: {found_zip}")
    print("Copying dataset to Colab local storage...")
    shutil.copy(found_zip, '/content/dataset.zip')

    print("Unzipping dataset...")
    os.makedirs('/content/dataset', exist_ok=True)
    !unzip -q /content/dataset.zip -d /content/dataset

    print("\n🎉 Success! Here is what is inside your dataset folder:")
    !ls -la /content/dataset
else:
    print("❌ Could not find the zip file automatically. Let's check what folders are in MyDrive:")
    !ls -la "/content/drive/MyDrive"

✅ Found your dataset at: /content/drive/MyDrive/Marine_dataset/archive 2.57.51 PM.zip
Copying dataset to Colab local storage...
Unzipping dataset...

🎉 Success! Here is what is inside your dataset folder:
total 12
drwxr-xr-x 3 root root 4096 Aug 15 09:37 .
drwxr-xr-x 1 root root 4096 Aug 15 09:37 ..
drwxr-xr-x 5 root root 4096 Aug 15 09:37 underwater_plastics


In [5]:
import os
print("Contents of underwater_plastics:")
!ls -la /content/dataset/underwater_plastics

Contents of underwater_plastics:
total 24
drwxr-xr-x 5 root root 4096 Aug 15 09:37 .
drwxr-xr-x 3 root root 4096 Aug 15 09:37 ..
-rw-r--r-- 1 root root  406 Mar 26  2024 data.yaml
drwxr-xr-x 4 root root 4096 Aug 15 09:37 test
drwxr-xr-x 4 root root 4096 Aug 15 09:37 train
drwxr-xr-x 4 root root 4096 Aug 15 09:37 valid


In [6]:
import cv2
import numpy as np
import os
import shutil
from pathlib import Path

def enhance_underwater_image(img):
    """
    Executes the exact preprocessing pipeline from your architecture diagram:
    - Color Correction
    - Dehazing (White-balance & CLAHE)
    - Image Enhancement
    """
    result = img.astype(np.float32)

    # --- 1. COLOR CORRECTION & WHITE-BALANCE (Dehazing refinement) ---
    # Using the Gray World assumption to eliminate the heavy blue/green cast
    avg_b = np.mean(result[:, :, 0])
    avg_g = np.mean(result[:, :, 1])
    avg_r = np.mean(result[:, :, 2])
    avg_gray = (avg_b + avg_g + avg_r) / 3 if (avg_b + avg_g + avg_r) > 0 else 1

    if avg_b > 0 and avg_g > 0 and avg_r > 0:
        result[:, :, 0] *= (avg_gray / avg_b) # Blue channel adjustment
        result[:, :, 1] *= (avg_gray / avg_g) # Green channel adjustment
        result[:, :, 2] *= (avg_gray / avg_r) # Red channel adjustment

    result = np.clip(result, 0, 255).astype(np.uint8)

    # --- 2. IMAGE ENHANCEMENT & DEHAZING (CLAHE) ---
    # Convert to LAB color space to process lighting/contrast separately from color
    lab = cv2.cvtColor(result, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    # Apply CLAHE (Contrast Limited Adaptive Histogram Equalization) to remove haze and boost visibility
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)

    # Merge channels back and convert back to BGR format
    merged_lab = cv2.merge((cl, a_channel, b_channel))
    enhanced_img = cv2.cvtColor(merged_lab, cv2.COLOR_LAB2BGR)

    return enhanced_img

# Define source and target preprocessed paths
source_dir = Path('/content/dataset/underwater_plastics')
target_dir = Path('/content/preprocessed_dataset')

# Create a clean copy of the dataset folder structure
if target_dir.exists():
    shutil.rmtree(target_dir)
shutil.copytree(source_dir, target_dir)

# Process all images located inside the dataset
image_extensions = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
count = 0

for img_path in target_dir.glob('**/*'):
    if img_path.suffix in image_extensions:
        img = cv2.imread(str(img_path))
        if img is not None:
            processed = enhance_underwater_image(img)
            cv2.imwrite(str(img_path), processed)
            count += 1

print(f"✅ Successfully preprocessed {count} images with Color Correction, White-Balance, and CLAHE Dehazing!")

✅ Successfully preprocessed 5130 images with Color Correction, White-Balance, and CLAHE Dehazing!


In [7]:
# Install Ultralytics if not already present
!pip install -q ultralytics

from ultralytics import YOLO

# Load the pre-trained YOLO11 medium model architecture
model = YOLO('yolo11m.pt')

# Start training on your preprocessed dataset
results = model.train(
    data='/content/preprocessed_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='yolo11m_preprocessed_marine'
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 6.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/preprocessed_dataset/data.ya

In [8]:
from ultralytics import YOLO

# Load your newly trained best weights
model = YOLO('/content/runs/detect/yolo11m_preprocessed_marine/weights/best.pt')

# Run validation and get the metrics object
metrics = model.val(data='/content/preprocessed_dataset/data.yaml')

# Print core accuracy numbers cleanly
print(f"🎯 mAP50: {metrics.box.map50:.4f}")
print(f"🎯 mAP50-95: {metrics.box.map:.4f}")
print(f"🎯 Precision: {metrics.box.p.mean():.4f}")
print(f"🎯 Recall: {metrics.box.r.mean():.4f}")

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,041,597 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1852.9±468.3 MB/s, size: 83.6 KB)
val: Scanning /content/preprocessed_dataset/valid/labels.cache... 1001 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1001/1001 199.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 1.8it/s 34.9s
                   all       1001       1891      0.787      0.684      0.735      0.487
                  Mask         77         90          1      0.418      0.768      0.567
                   can         18         20      0.834       0.65      0.647      0.291
             cellphone         61         71      0.954      0.972      0.975      0.865
           electronics         27         40      0.927      0.575      0.682      0.409
               gbottle   

In [9]:
from google.colab import files

# Download the best model weights
files.download('/content/runs/detect/yolo11m_preprocessed_marine/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>